In [1]:
import os
if os.name == 'nt':
    os.add_dll_directory(r'C:\Program Files\SuperTuxKart 1.5')

import multiprocessing as mp
import numpy as np

import torch
import torch.optim as optim
import torch.nn.functional as F
from actor import ActorNetwork
from critic import CriticNetwork
from env_worker import SingleInstance


In [2]:
actor_net  = ActorNetwork(state_dim=52)
critic_net = CriticNetwork(state_dim=52)
RESUME_TRAINING = True
if RESUME_TRAINING and os.path.exists('best_actor.pth') and os.path.exists('best_critic.pth'):
    try:
        actor_net.load_state_dict(torch.load('best_actor.pth', map_location='cpu'))
        critic_net.load_state_dict(torch.load('best_critic.pth', map_location='cpu'))
        print('Loaded weights from best_actor.pth & best_critic.pth resuming training!')
    except Exception as e:
        print(f'Starting from scratch: {e}')
else:
    print('Starting training from scratch with random weights.')

actor_net.eval()
critic_net.eval()

optimizer = optim.Adam([
    {'params': actor_net.parameters(),  'lr': 3e-4},
    {'params': critic_net.parameters(), 'lr': 1e-3},
])

def compute_gae(rewards, values, dones, gamma=0.995, lam=0.95):
    advantages = []
    gae        = 0.0
    next_value = 0.0
    for t in reversed(range(len(rewards))):
        mask   = 0.0 if dones[t] else 1.0
        delta  = rewards[t] + gamma * next_value * mask - values[t]
        gae    = delta + gamma * lam * mask * gae
        advantages.insert(0, gae)
        next_value = values[t]
    advantages = torch.FloatTensor(advantages).unsqueeze(1)
    returns    = advantages + torch.FloatTensor(values).unsqueeze(1)
    return advantages, returns

def update_ppo(states, actions, advantages, returns, old_log_probs,
               epochs=6, clip_epsilon=0.2, entropy_coeff=0.01,
               value_coeff=0.5, max_grad_norm=0.5):
    actor_net.train()
    critic_net.train()
    for epoch in range(epochs):
        if torch.isnan(states).any():
            print('NaN detected in states buffer! Skipping PPO update.')
            break
        action_dists   = actor_net(states)
        new_values     = critic_net(states)
        new_log_probs  = action_dists.log_prob(actions).sum(dim=-1, keepdim=True)
        entropy        = action_dists.entropy().sum(dim=-1, keepdim=True).mean()
        ratio  = torch.exp(new_log_probs - old_log_probs)
        surr1  = ratio * advantages
        surr2  = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * advantages
        actor_loss  = -torch.min(surr1, surr2).mean()
        critic_loss = F.mse_loss(new_values, returns)
        loss        = actor_loss + value_coeff * critic_loss - entropy_coeff * entropy
        optimizer.zero_grad()
        loss.backward()
        actor_grad_norm  = torch.nn.utils.clip_grad_norm_(actor_net.parameters(),  max_grad_norm)
        critic_grad_norm = torch.nn.utils.clip_grad_norm_(critic_net.parameters(), max_grad_norm)
        optimizer.step()
    actor_net.eval()
    critic_net.eval()
    with torch.no_grad():
        final_log_probs = actor_net(states).log_prob(actions).sum(dim=-1, keepdim=True)
        kl_approx = (old_log_probs - final_log_probs).mean().item()
    print(f'  [PPO] actor_loss={actor_loss.item():.4f}  '
          f'critic_loss={critic_loss.item():.4f}  '
          f'entropy={entropy.item():.4f}  '
          f'approx_kl={kl_approx:.4f}  '
          f'actor_grad={actor_grad_norm:.4f}')


Loaded weights from best_actor.pth & best_critic.pth resuming training!


In [ ]:
def main():
    PATIENCE_LIMIT    = 60
    NUM_EPISODES      = 500
    NUM_WORKERS       = 5
    STEPS_PER_EPISODE = 1000

    best_reward      = float('-inf')
    patience_counter = 0

    for episode in range(NUM_EPISODES):
        worker_buffers       = [[] for _ in range(NUM_WORKERS)]
        total_episode_reward = 0.0
        ProcessList          = []
        ConList              = []

        for i in range(NUM_WORKERS):
            ParentCon, ChildCon = mp.Pipe()
            process = mp.Process(target=SingleInstance, args=(i, ChildCon))
            ProcessList.append(process)
            ConList.append(ParentCon)
            process.start()

        try:
            BatchStates = [None] * NUM_WORKERS
            BatchDones  = [False] * NUM_WORKERS
            for i, con in enumerate(ConList):
                try:
                    np_obs, reward, RaceDone = con.recv()
                    BatchStates[i] = np_obs
                    BatchDones[i]  = bool(RaceDone)
                except (EOFError, BrokenPipeError, OSError):
                    print(f'  [WARN] Worker {i} pipe broken during startup.')
                    BatchStates[i] = np.zeros(52, dtype=np.float32)
                    BatchDones[i]  = True

            alive_count = sum(1 for d in BatchDones if not d)
            if alive_count == 0:
                print('  All workers died on startup! Skipping episode.')
                continue

            for step in range(STEPS_PER_EPISODE):
                active_states = [BatchStates[i] for i in range(NUM_WORKERS) if not BatchDones[i]]
                if not active_states:
                    break

                state_tensor = torch.FloatTensor(np.array(BatchStates))
                if torch.isnan(state_tensor).any():
                    print('NaN in engine observations! Terminating episode.')
                    break

                with torch.no_grad():
                    action_dist    = actor_net(state_tensor)
                    sampled_action = action_dist.sample()
                    state_value    = critic_net(state_tensor)
                    BatchLogProbs  = action_dist.log_prob(sampled_action).sum(dim=-1)

                MemoryActions = []
                for i in range(NUM_WORKERS):
                    steer_val = float(torch.clamp(sampled_action[i, 0], -1.0,  1.0).item())
                    accel_val = float(torch.clamp(sampled_action[i, 1],  0.0,  1.0).item())
                    brake_val = float(torch.clamp(sampled_action[i, 2],  0.0,  1.0).item())
                    MemoryActions.append((steer_val, accel_val, brake_val))

                for i, con in enumerate(ConList):
                    if not BatchDones[i]:
                        try:
                            con.send(MemoryActions[i])
                        except (EOFError, BrokenPipeError, OSError):
                            BatchDones[i] = True

                for i, con in enumerate(ConList):
                    if not BatchDones[i]:
                        try:
                            np_obs, reward, RaceDone = con.recv()
                            worker_buffers[i].append({
                                'state':    BatchStates[i],
                                'action':   sampled_action[i].cpu().numpy(),
                                'reward':   float(reward),
                                'value':    state_value[i].item(),
                                'log_prob': BatchLogProbs[i].item(),
                                'done':     bool(RaceDone),
                            })
                            BatchStates[i] = np_obs
                            total_episode_reward += float(reward)
                            if RaceDone:
                                BatchDones[i] = True
                        except (EOFError, BrokenPipeError, OSError):
                            BatchDones[i] = True

                if all(BatchDones):
                    print(f'  All workers finished at step {step + 1}')
                    break

            total_transitions = sum(len(b) for b in worker_buffers)
            mean_reward = total_episode_reward / max(NUM_WORKERS, 1)
            print(f'Episode {episode + 1}/{NUM_EPISODES} | '
                  f'Mean Reward: {mean_reward:.3f} | '
                  f'Transitions: {total_transitions}')

            all_states     = []
            all_actions    = []
            all_advantages = []
            all_returns    = []
            all_log_probs  = []

            for w_buf in worker_buffers:
                if not w_buf:
                    continue
                w_rewards = [t['reward'] for t in w_buf]
                w_values  = [t['value']  for t in w_buf]
                w_dones   = [t['done']   for t in w_buf]
                w_adv, w_ret = compute_gae(w_rewards, w_values, w_dones)

                all_states.append(torch.FloatTensor(np.array([t['state'] for t in w_buf])))
                all_actions.append(torch.FloatTensor(np.array([t['action'] for t in w_buf])))
                all_advantages.append(w_adv)
                all_returns.append(w_ret)
                all_log_probs.append(torch.FloatTensor([t['log_prob'] for t in w_buf]).unsqueeze(1))

            if all_states:
                batch_states     = torch.cat(all_states, dim=0)
                batch_actions    = torch.cat(all_actions, dim=0)
                batch_advantages = torch.cat(all_advantages, dim=0)
                batch_returns    = torch.cat(all_returns, dim=0)
                batch_log_probs  = torch.cat(all_log_probs, dim=0)

                batch_advantages = (batch_advantages - batch_advantages.mean()) / (batch_advantages.std(unbiased=False) + 1e-8)
                update_ppo(batch_states, batch_actions, batch_advantages, batch_returns, batch_log_probs)

            if mean_reward > best_reward:
                best_reward      = mean_reward
                patience_counter = 0
                torch.save(actor_net.state_dict(),  'best_actor.pth')
                torch.save(critic_net.state_dict(), 'best_critic.pth')
                print(f'New best: {best_reward:.4f} models saved.')
            else:
                patience_counter += 1
                print(f'  No improvement. Patience: {patience_counter}/{PATIENCE_LIMIT}')

            if patience_counter >= PATIENCE_LIMIT:
                print(f'Early stopping after {episode + 1} episodes.')
                break
        finally:
            for con in ConList:
                try:
                    con.send('TERMINATE')
                except Exception:
                    pass
            for process in ProcessList:
                process.join(timeout=3)
                if process.is_alive():
                    process.terminate()


In [4]:
if __name__ == '__main__':
    main()


  All workers finished at step 459
Episode 1/500 | Mean Reward: 68466.067 | Transitions: 2288
  [PPO] actor_loss=0.0079  critic_loss=836211.9375  entropy=2.9443  approx_kl=0.0237  actor_grad=1.3363
  âœ“ New best: 68466.0666 â€” models saved.
  All workers finished at step 473
Episode 2/500 | Mean Reward: 68445.337 | Transitions: 2313
  [PPO] actor_loss=-0.0012  critic_loss=489813.2812  entropy=2.9453  approx_kl=0.0102  actor_grad=0.5629
  No improvement. Patience: 1/60
  All workers finished at step 464
Episode 3/500 | Mean Reward: 68439.119 | Transitions: 2288
  [PPO] actor_loss=-0.0006  critic_loss=348093.5938  entropy=2.9457  approx_kl=-0.0012  actor_grad=0.1506
  No improvement. Patience: 2/60
  All workers finished at step 467
Episode 4/500 | Mean Reward: 68453.666 | Transitions: 2300
  [PPO] actor_loss=-0.0010  critic_loss=395013.7500  entropy=2.9486  approx_kl=0.0077  actor_grad=0.2032
  No improvement. Patience: 3/60
  All workers finished at step 461
Episode 5/500 | Mean Rewa